# Inspeção visual do Kvasir-SEG

Este notebook verifica visualmente se o pré-processamento das máscaras e sua conversão para o formato **YOLO Segmentation** estão corretos. Vamos comparar cinco imagens com suas máscaras originais, máscaras binárias processadas e polígonos reconstruídos a partir dos labels já existentes.

Todas as operações são feitas em memória: nenhum arquivo do dataset é criado, alterado ou movido. Execute as células em ordem, usando o ambiente Python do projeto.

In [ ]:
from pathlib import Path
import random

import cv2
import numpy as np
import matplotlib.pyplot as plt

: 

## 1. Localizar os dados

O notebook fica em `notebooks/`. Para funcionar também quando o Jupyter é iniciado na raiz do projeto, procuramos `pyproject.toml` no diretório atual e em seus diretórios pais. Inicie o notebook na raiz do projeto ou em uma subpasta dela.

In [ ]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = None

for candidate in (cwd, *cwd.parents):
    if (candidate / "pyproject.toml").is_file():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Não foi possível localizar pyproject.toml a partir de {cwd}. "
        "Inicie o Jupyter na raiz do projeto ou na pasta notebooks/."
    )

raw_path = PROJECT_ROOT / "data/raw/Kvasir-SEG"
processed_path = PROJECT_ROOT / "data/processed/Kvasir-SEG"

images_path = raw_path / "images"
masks_path = raw_path / "masks"
binary_masks_path = processed_path / "masks_binary"
labels_path = processed_path / "labels_all"

for directory in (raw_path, processed_path, images_path, masks_path, binary_masks_path, labels_path):
    if not directory.is_dir():
        raise FileNotFoundError(f"Diretório necessário não encontrado: {directory}")

print(f"Raiz do projeto: {PROJECT_ROOT}")

## 2. Selecionar cinco imagens

Ordenamos os arquivos antes do sorteio e usamos a semente `42` para repetir a mesma seleção quando o dataset não mudar.

In [ ]:
image_files = sorted(images_path.glob("*.jpg"))
if len(image_files) < 5:
    raise ValueError(
        f"São necessárias pelo menos 5 imagens .jpg em {images_path}; "
        f"foram encontradas {len(image_files)}."
    )

random.seed(42)
selected_images = random.sample(image_files, 5)

print("Imagens selecionadas:")
for image_path in selected_images:
    print(f"- {image_path.name}")

## 3. Comparar as máscaras

A máscara original tem o mesmo nome da imagem (`.jpg`); a máscara processada usa o mesmo nome-base com extensão `.png`. O pré-processamento do projeto salva valores **0 e 1**, por isso a máscara processada é exibida com `vmin=0` e `vmax=1`.

Convertemos a imagem de BGR para RGB e lemos as duas máscaras em escala de cinza. Guardamos as amostras em memória para reutilizá-las nas próximas comparações. Arquivos ausentes, ilegíveis ou com dimensões incompatíveis geram erros com o nome do arquivo.

In [ ]:
samples = []

for image_path in selected_images:
    mask_path = masks_path / image_path.name
    binary_mask_path = binary_masks_path / f"{image_path.stem}.png"

    for description, file_path in (
        ("Imagem original", image_path),
        ("Máscara original correspondente", mask_path),
        ("Máscara processada", binary_mask_path),
    ):
        if not file_path.is_file():
            raise FileNotFoundError(f"{description} ausente: {file_path}")

    image_bgr = cv2.imread(str(image_path))
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    binary_mask = cv2.imread(str(binary_mask_path), cv2.IMREAD_GRAYSCALE)

    for array, file_path in (
        (image_bgr, image_path), (mask, mask_path), (binary_mask, binary_mask_path)
    ):
        if array is None:
            raise ValueError(f"Não foi possível ler o arquivo: {file_path}")
        if array.shape[:2] != image_bgr.shape[:2]:
            raise ValueError(f"Dimensões diferentes da imagem original: {file_path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    samples.append({
        "image_path": image_path,
        "image": image_rgb,
        "mask": mask,
        "binary_mask": binary_mask,
    })

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image_rgb)
    axes[1].imshow(mask, cmap="gray", vmin=0, vmax=255)
    axes[2].imshow(binary_mask, cmap="gray", vmin=0, vmax=1)
    for ax, title in zip(axes, ("Imagem original", "Máscara original", "Máscara processada")):
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(image_path.name)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## 4. Ler os polígonos YOLO Segmentation

Cada linha do label gerado por `convert_segment_masks_to_yolo_seg` segue o formato:

```text
class_id x1 y1 x2 y2 x3 y3 ...
```

A função recebe o caminho do label e as dimensões `(altura, largura)` da imagem. Retorna uma lista de pares `(class_id, pontos_em_pixels)`, um por segmento. Multiplicamos as coordenadas normalizadas pela largura e pela altura, arredondamos e limitamos os pontos à área da imagem.

Linhas em branco são ignoradas. Um label ausente gera uma exceção; um label vazio gera um aviso e retorna uma lista vazia. Linhas malformadas geram um erro que identifica o arquivo e a linha.

In [ ]:
def read_yolo_segments(label_path, image_shape):
    """Retorna [(class_id, pontos Nx2 em pixels), ...] para (altura, largura)."""
    label_path = Path(label_path)
    if not label_path.is_file():
        raise FileNotFoundError(f"Label YOLO ausente: {label_path}")

    height, width = image_shape[:2]
    if height <= 0 or width <= 0:
        raise ValueError(f"Dimensões de imagem inválidas: {image_shape}")

    segments = []
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        fields = line.split()
        if not fields:
            continue

        try:
            class_id = int(fields[0])
            coordinates = np.array(fields[1:], dtype=np.float64)
        except ValueError as error:
            raise ValueError(
                f"Label inválido em {label_path}, linha {line_number}: "
                "esperados class_id inteiro e coordenadas numéricas."
            ) from error

        if class_id < 0 or coordinates.size < 6 or coordinates.size % 2 != 0:
            raise ValueError(
                f"Label inválido em {label_path}, linha {line_number}: "
                "esperados class_id não negativo e pelo menos 3 pares (x, y)."
            )
        if not np.all(np.isfinite(coordinates)) or np.any((coordinates < 0) | (coordinates > 1)):
            raise ValueError(
                f"Coordenadas inválidas em {label_path}, linha {line_number}: "
                "os valores devem ser finitos e estar entre 0 e 1."
            )

        points = coordinates.reshape(-1, 2) * np.array([width, height])
        points = np.rint(points).astype(np.int32)
        points[:, 0] = np.clip(points[:, 0], 0, width - 1)
        points[:, 1] = np.clip(points[:, 1], 0, height - 1)
        segments.append((class_id, points))

    if not segments:
        print(f"Aviso: label YOLO vazio, sem polígonos para exibir: {label_path}")

    return segments

## 5. Sobrepor os polígonos à imagem

Reutilizamos cada imagem RGB carregada anteriormente e desenhamos os contornos em verde sobre uma **cópia em memória**, usando `cv2.polylines`. Todos os segmentos do label são desenhados. O título informa quando o label está vazio.

In [ ]:
for sample in samples:
    image_path = sample["image_path"]
    image_rgb = sample["image"]
    label_path = labels_path / f"{image_path.stem}.txt"
    segments = read_yolo_segments(label_path, image_rgb.shape[:2])

    overlay = image_rgb.copy()
    for class_id, points in segments:
        cv2.polylines(
            overlay, [points.reshape(-1, 1, 2)], isClosed=True,
            color=(0, 255, 0), thickness=2,
        )

    sample["yolo_image"] = overlay
    sample["yolo_title"] = (
        f"Polígono YOLO ({len(segments)} segmento(s))"
        if segments else "Polígono YOLO — label vazio"
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(overlay)
    ax.set_title(f"{image_path.name}\n{sample['yolo_title']}")
    ax.axis("off")
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## 6. Comparação final

Para cada imagem, compare as quatro etapas lado a lado. Observe se o contorno verde acompanha o pólipo anotado nas máscaras e se o pré-processamento preservou sua região. Esta inspeção é visual e não calcula métricas de segmentação.

In [ ]:
for sample in samples:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(sample["image"])
    axes[1].imshow(sample["mask"], cmap="gray", vmin=0, vmax=255)
    axes[2].imshow(sample["binary_mask"], cmap="gray", vmin=0, vmax=1)
    axes[3].imshow(sample["yolo_image"])

    titles = ("Imagem original", "Máscara original", "Máscara processada", sample["yolo_title"])
    for ax, title in zip(axes, titles):
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(sample["image_path"].name)
    fig.tight_layout()
    plt.show()
    plt.close(fig)